# TN3 — Cross-category: train trên sản phẩm A, dùng cho sản phẩm B (Google Colab)

**Câu hỏi:** model của sản phẩm A dùng được bao nhiêu cho sản phẩm B? Đổi sang mã hàng mới thì cần thêm bao nhiêu ảnh?

| Phần | Dữ liệu | Ý nghĩa |
|---|---|---|
| **TN3a** | VisA `pcb1`–`pcb4`, ma trận 4×4 | pcb1, pcb2 là hai mặt của cùng một bo; pcb3, pcb4 là bo khác trên nền khác |
| **TN3a-thích nghi** | pcb1 → pcb2, pcb3, pcb4 | Thêm k = 4 hoặc 16 ảnh good của bo đích; so với chỉ train trên k ảnh đó |
| **TN3b** | MVTec AD screw, grid, transistor, ma trận 3×3 | Đối chứng: các category hoàn toàn khác nhau |

Test luôn trên test set của category **đích**. Ngưỡng = trung bình + 3 độ lệch chuẩn trên 15% ảnh train good của nguồn (tách riêng, không dùng để train). Thời gian trên T4: TN3a khoảng 40–60 phút, thích nghi 25–35 phút, TN3b 15 phút.

## 1. Cài đặt
Kết nối Drive, lấy code từ repo (nhánh `an/tn-refactor`) và kiểm tra GPU. Sau một lỗi CUDA phải *Disconnect and delete runtime* rồi chạy lại từ đầu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
REPO_URL = 'https://github.com/AnDpTri/mvtec-anomaly-detection.git'
BRANCH = 'an/tn-refactor'
if subprocess.run(['git', '-C', '/content/repo', 'pull', '-q'], capture_output=True).returncode != 0:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'], check=True)
sys.path.insert(0, '/content/repo')

from src.tn.config import check_gpu
check_gpu()

## 2. Cấu hình

In [ ]:
from pathlib import Path
from src.tn.results import Results

DRIVE_ROOT = Path('/content/drive/MyDrive/MVtecAD_Deep_Learning')
DATA_ROOT  = Path('/content/data/MVTec_AD')                 # chép ra ổ local của Colab (đọc nhanh hơn Drive)
RESULTS    = DRIVE_ROOT / 'results'
MAPS_DIR   = RESULTS / 'maps'
FIG_DIR    = RESULTS / 'figures' / 'tn3'
for p in (RESULTS, MAPS_DIR, FIG_DIR): p.mkdir(parents=True, exist_ok=True)
res = Results(RESULTS)
VISA_DIR   = Path('/content/data/VisA')

METHODS    = ['padim', 'patchcore', 'ae']
CATEGORIES = ['screw', 'grid', 'transistor']        # TN3b

VISA_PCBS    = ['pcb1', 'pcb2', 'pcb3', 'pcb4']
VISA_HW      = (256, 320)                           # chung cho 4 bo (PaDiM cần nguồn/đích cùng kích thước)
VISA_CORESET = 0.01                                 # ~900 ảnh train -> bank rất lớn -> coreset 1%

ADAPT_SOURCE  = 'pcb1'
ADAPT_TARGETS = ['pcb2', 'pcb3', 'pcb4']
ADAPT_K       = [4, 16]
ADAPT_SEEDS   = [0]                                 # tăng lên [0, 1, 2] nếu còn thời gian GPU

## 3. Dữ liệu
MVTec AD từ `mvtec_3cat.zip` trên Drive; VisA tải thẳng từ AWS (~1,8 GB).

In [ ]:
from functools import partial
from src.tn import data as D

D.prepare_mvtec(DATA_ROOT, CATEGORIES, DRIVE_ROOT / 'mvtec_3cat.zip')
VISA_ROOT, VISA_SPLIT = D.prepare_visa(VISA_DIR, VISA_PCBS)
load_visa  = partial(D.load_visa, VISA_ROOT, VISA_SPLIT)
load_mvtec = partial(D.load_mvtec, DATA_ROOT)

## 4. TN3a: ma trận 4×4 trên VisA PCB

In [ ]:
from src.tn.config import require_gpu
from src.tn.experiments import run_matrix, run_adapt

require_gpu()
run_matrix(res, 'tn3_visa', load_visa, VISA_PCBS, METHODS, hw=VISA_HW, coreset=VISA_CORESET)

## 5. TN3a-thích nghi: thêm k ảnh good của bo đích
**nguồn + k:** ảnh pcb1 cộng k ảnh good của bo đích ("đổi mã hàng, tranh thủ chụp vài ảnh mới"). **chỉ k:** chỉ k ảnh đó, làm mốc so sánh. Ngưỡng đặt từ 15% ảnh good của bo đích.

In [ ]:
require_gpu()
run_adapt(res, load_visa, ADAPT_SOURCE, ADAPT_TARGETS, ADAPT_K, ADAPT_SEEDS, METHODS, VISA_HW, coreset=VISA_CORESET)

## 6. TN3b: ma trận 3×3 trên MVTec AD (đối chứng)

In [ ]:
require_gpu()
run_matrix(res, 'tn3_mvtec', load_mvtec, CATEGORIES, METHODS)

## 7. Phân tích

In [ ]:
from src.tn import analysis as A

for exp, cats, title in [('tn3_visa', VISA_PCBS, 'TN3a — VisA PCB: train trên bo NGUỒN, test trên bo ĐÍCH'),
                         ('tn3_mvtec', CATEGORIES, 'TN3b — MVTec AD: train trên category NGUỒN, test trên category ĐÍCH')]:
    df = res.load(exp)
    if not len(df): print('Chưa có kết quả', exp); continue
    df.to_csv(RESULTS / f'{exp}_summary.csv', index=False)
    A.plot_matrices(df, cats, title, FIG_DIR / f'{exp}_matrix.png')
    display(A.diagonal_summary(df))

In [ ]:
ad, mx = res.load('tn3_adapt'), res.load('tn3_visa')
if len(ad):
    ad.to_csv(RESULTS / 'tn3_adapt_summary.csv', index=False)
    A.plot_adapt(ad, mx, ADAPT_SOURCE, ADAPT_TARGETS, ADAPT_K, FIG_DIR / 'tn3_adapt.png')
    display(ad.pivot_table(index=['target', 'k'], columns=['regime', 'method'], values='i_auroc').round(3))